SWING EQUATION MODELING

The swing equation describes the relation between the rotational speed and torque. It can be expressed as J*dw/dt = Tm-Tl, where H´J is equal to the moment of inertia, w is equal to the rotational speed, Tm is equal to the machine torque and Tl is equal to the load torque. Based on the book, we can model this equation in the electrical domain by defining J as the capacitance to the ground (C), w as the node voltage (V), and torque as the current going into a node (I). In this way, we get the equation that describes the relation between the voltage and current of a capacitor. 

In [1]:
from villas.dataprocessing.readtools import *
from villas.dataprocessing.timeseries import *
from villas.dataprocessing.timeseries import TimeSeries as ts
import matplotlib.pyplot as plt
import numpy as np
import dpsimpy
import re

#%matplotlib widget

epsilon = 1e-12

In [2]:
# Nodes
gnd = dpsimpy.emt.SimNode.gnd
n0  = dpsimpy.emt.SimNode("n0")

# Parameters
i_s = dpsimpy.emt.ph1.CurrentSource("i_s")
i_s.set_parameters(complex(2.5, 0))
c = dpsimpy.emt.ph1.Capacitor("c")
c.C = 250e-3

# Connections
i_s.connect([gnd, n0])
c.connect([n0, gnd])

In [3]:
sys = dpsimpy.SystemTopology(50, [ n0 ], [ i_s, c ])

In [ ]:
sim = dpsimpy.Simulation("Swing_equation", loglevel=dpsimpy.LogLevel.debug)
sim.set_system(sys)
sim.set_domain(dpsimpy.Domain.EMT)
sim.set_time_step(0.0001)
sim.set_final_time(10)

log = dpsimpy.Logger("Swing_equation")
for i in range(0, len(sys.nodes)):
    log.log_attribute("v" + str(i), "v", sys.nodes[i])

sim.add_logger(log)
    
sim.run()

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
%config InlineBackend.rc = {'font.size': 10, 'figure.figsize': (6.0, 4.0), 'figure.facecolor': 'white', 'savefig.dpi': 72, 'figure.subplot.bottom': 0.125, 'figure.edgecolor': 'white'}

import matplotlib.pyplot as plt
import villas.dataprocessing.plottools as pt
import villas.dataprocessing.readtools as rt
import villas.dataprocessing.timeseries as ts

results     = rt.read_timeseries_dpsim('logs/Swing_equation.csv')
results_emt = [ results[series].frequency_shift(freq=0) for series in results ]

#for series in results_emt:
    #pt.plot_timeseries('Results EMT', series)
pt.plot_timeseries('Results EMT', results_emt[0])
plt.show()